# Label and Define Each Cluster

| Item | Description |
|---|---|
| Goal | Ask the model to draft a codebook entry for each of the 150 topic clusters of T2.1.2: a short neutral label, a one-sentence definition, notes on how the topic differs from its two nearest clusters, and five example phrases copied from the cluster. The model sees only member phrases, so it names what is there and cannot add a topic. The draft is what T2.1.4 reviews. |
| Input | `phrases.parquet`, `cut_labels.parquet`, `cluster_stats.parquet`, and `unit_embeddings.npz` in `taxonomy/` of the T2.1 Interim folder; the model Qwen/Qwen3.5-4B from the project cache. |
| Output | `taxonomy/codebook_draft.parquet` and `taxonomy/status/codebook_draft.jsonl` in the T2.1 Interim folder; the draft as a readable document, `table_codebook_draft.md`, with a CSV copy, under `R - Results/tables/...`. |
| Variables | • `kind`: `topic` for the 150 topic clusters, `theme` for the 15 broad clusters that were drafted at the same time and are kept only for the record<br>• `entry`: the parsed draft with `label`, `definition`, `exclusion_notes`, `exemplars`, and `label_lint`<br>• `stability`, `cohesion`: the cluster diagnostics of T2.1.2, carried along for the review |
| Run | The drafting ran once on 2026-08-22 on one A40; its outputs are shown. Not rerun: the review in T2.1.4 was made on this draft, and a new run would word the labels differently. The document and the check in Steps 7 and 8 were run again on 2026-10-31 from the stored draft. |


## Step 1. Import packages


In [ ]:
import hashlib
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


## Step 2. Define file locations


In [ ]:
PROJECT_FOLDER = Path.cwd().resolve().parents[2]  # <pipeline>/<section>/<subfolder>/ sits three levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T - Three Step Text Analysis Pipeline/T2 - Topic Extraction and Stance Analysis/T2.1 - Track 1 - Topic Extraction based on Hearing Summaries")
NOTEBOOK_NAME = "T2.1.3 - Draft - Label and Define Each Cluster"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
TAXONOMY_FOLDER = INTERIM_FOLDER / "taxonomy"
K_TOPIC, K_THEME = 150, 15  # the topic count set for a granularity comparable with CAP's subtopics; see Step 5
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T - Three Step Text Analysis Pipeline/T2 - Topic Extraction and Stance Analysis" / ".env"
for folder in (INTERIM_FOLDER, PROCESSED_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(MODEL_CACHE)
# A token is needed only for gated models. Qwen3.5-4B and bge-large-en-v1.5 are open; the lines stay so the cell also serves a gated model.
if ENV_FILE.exists():
    for line in ENV_FILE.read_text().splitlines():
        name, _, value = line.strip().partition("=")
        if name == "HF_TOKEN" and value:
            os.environ["HF_TOKEN"] = value.strip().strip('"').strip("'")


## Step 3. Define the prompt

The model receives the cluster's most frequent phrases with their mention counts and the top phrases of the two nearest clusters. It returns a JSON object with the four fields. Example phrases must be copied verbatim; the parser drops any that are not. The label passes the same directional lint as the topic phrases.


In [ ]:
CODEBOOK_PROMPT_VERSION = "codebook-draft-0.1"
MAX_NEW_TOKENS = 512

CB_SYSTEM = ("You are helping construct a topic codebook for the analysis of United States congressional hearings. "
             "You use only the phrases provided; you never invent content.")

CB_USER = """Below are the member phrases of one topic cluster (most frequent first), followed by the top phrases of its nearest neighboring clusters.

CLUSTER MEMBERS (phrase — mentions):
{members}

NEIGHBOR 1 (a different topic): {n1}
NEIGHBOR 2 (a different topic): {n2}

Task: draft the codebook entry for this cluster.
- "label": a short, specific, neutral noun phrase naming the topic. Never directional wording (no ban/support/expand/protect); name the policy object, not a stance toward it.
- "definition": one sentence stating what belongs in this topic.
- "exclusion_notes": one or two short notes on how this topic differs from the two neighbors shown.
- "exemplars": exactly 5 phrases copied VERBATIM from the CLUSTER MEMBERS list above (or all of them, if fewer than 5 are listed).

Return only a JSON object with exactly those four fields. Your very first output character must be {{."""

# Directional words that must not appear inside a topic phrase; direction belongs in the stance field.
# The lint flags a phrase for review. It does not rewrite it.
DIRECTIONAL_LINT = re.compile(
    r"\b(?:ban(?:ning)?|block(?:ing)?|stop(?:ping)?|halt(?:ing)?|"
    r"oppos(?:e|ing|ition)|support(?:ing)?|defend(?:ing)?|protect(?:ing|ion)?|"
    r"against|anti|pro|promot(?:e|ing|ion)|expand(?:ing)?|restrict(?:ing|ion)?|"
    r"repeal(?:ing)?|eliminat(?:e|ing|ion))\b",
    re.IGNORECASE,
)


def lint_topic(phrase):
    """The directional word a topic phrase contains, or None if clean."""
    match = DIRECTIONAL_LINT.search(phrase or "")
    return match.group(0) if match else None


## Step 4. Define the model call, the log, and the parser


In [ ]:
GENERATION = {"do_sample": True, "temperature": 0.1}  # near-greedy sampling, the same setting as T1


def render(tokenizer, system, user):
    """Apply the model's chat template, with the thinking mode off where the template offers it."""
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate(model, tokenizer, system, user, max_new_tokens):
    """One prompt, one completion, fresh context. Also returns whether the completion ran into the token cap."""
    prompt = render(tokenizer, system, user)
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    n_in = ids["input_ids"].shape[-1]
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id, **GENERATION)
    generated = out[0][n_in:]
    hit_cap = len(generated) >= max_new_tokens
    return tokenizer.decode(generated, skip_special_tokens=True).strip(), hit_cap


class CallLog:
    """One JSON line per model call, written as it happens: stage, level, position, a hash of the exact input, token counts, time, and the full output. The same format as T1."""

    def __init__(self, path, document_id, model_name, prompt_version):
        self.path = Path(path) if path else None
        self.document_id = document_id
        self.model_name = model_name
        self.prompt_version = prompt_version
        self.records = []
        if self.path:
            self.path.parent.mkdir(parents=True, exist_ok=True)

    def write(self, stage, level, index, n_inputs, prompt_text, output, tokenizer, seconds):
        record = {
            "document_id": self.document_id, "model": self.model_name, "prompt_version": self.prompt_version,
            "stage": stage, "level": level, "index": index, "n_inputs": n_inputs,
            "input_sha256": hashlib.sha256(prompt_text.encode("utf-8")).hexdigest(),
            "input_chars": len(prompt_text),
            "input_tokens": len(tokenizer.encode(prompt_text, add_special_tokens=False)),
            "output_tokens": len(tokenizer.encode(output, add_special_tokens=False)),
            "output_chars": len(output), "seconds": round(seconds, 2),
            "at": datetime.now(timezone.utc).isoformat(timespec="seconds"), "output": output,
        }
        self.records.append(record)
        if self.path:
            with self.path.open("a") as log_file:
                log_file.write(json.dumps(record) + "\n")
                log_file.flush()


def parse_entry(output, members_cf):
    """Validate one drafted entry; example phrases that are not verbatim members are dropped and counted."""
    errors = []
    text = output.strip()
    if text.startswith("```"):
        errors.append("markdown_fence")
        text = re.sub(r"^```[a-zA-Z]*\s*|\s*```$", "", text).strip()
    try:
        raw = json.loads(text)
    except json.JSONDecodeError:
        return None, errors + ["json_parse"]
    if not isinstance(raw, dict):
        return None, errors + ["not_object"]
    label = str(raw.get("label", "")).strip()
    definition = str(raw.get("definition", "")).strip()
    notes = raw.get("exclusion_notes", [])
    if isinstance(notes, str):
        notes = [notes]
    ex = [str(e).strip() for e in raw.get("exemplars", [])]
    verbatim = [e for e in ex if e.casefold() in members_cf]
    if len(verbatim) < len(ex):
        errors.append(f"exemplars_not_verbatim:{len(ex) - len(verbatim)}")
    if not label or not definition:
        return None, errors + ["missing_field"]
    return {"label": label, "definition": definition, "exclusion_notes": [str(x) for x in notes][:2],
            "exemplars": verbatim[:5], "label_lint": lint_topic(label)}, errors


## Step 5. Load the clusters at the operating point

T2.1.2 scored nine candidate counts, and its criteria were flat across them. The topic level uses 150 clusters, a count set so that the candidates sit at a granularity comparable with CAP's subtopics, the yardstick of T2.3. The review in T2.1.4 then reduces them to 111. The 15-cluster level groups the 150 topics by the plurality of their phrase weight. It was drafted too and is kept for the record. The analysis groups the topics by hand in T2.1.5.


In [5]:
phrases = pd.read_parquet(TAXONOMY_FOLDER / "phrases.parquet")
cuts = pd.read_parquet(TAXONOMY_FOLDER / "cut_labels.parquet")
stats = pd.read_parquet(TAXONOMY_FOLDER / "cluster_stats.parquet")

ph = phrases[phrases["family_id"] >= 0].merge(cuts[["family_id", f"K{K_TOPIC}", f"K{K_THEME}"]], on="family_id")
ph = ph.rename(columns={f"K{K_TOPIC}": "topic_c", f"K{K_THEME}": "theme_c"})
stab_t = stats.query("K == @K_TOPIC").set_index("cluster")
stab_h = stats.query("K == @K_THEME").set_index("cluster")

nest = (ph.groupby(["topic_c", "theme_c"])["weight"].sum().reset_index()
          .sort_values("weight", ascending=False).drop_duplicates("topic_c").set_index("topic_c"))
share = nest["weight"] / ph.groupby("topic_c")["weight"].sum()
print(f"{K_TOPIC} topics / {K_THEME} themes; topic->theme plurality share: median {share.median():.0%}, min {share.min():.0%}")


150 topics / 15 themes; topic->theme plurality share: median 77%, min 19%


## Step 6. Draft every entry

Every cluster is drafted, not only the stable ones. Each call seeds the sampler from the cluster id and is logged with the prompt version.


In [2]:
MODEL = "Qwen/Qwen3.5-4B"
dat = np.load(TAXONOMY_FOLDER / "unit_embeddings.npz", allow_pickle=True)
X = dat["X"].astype(np.float32)
nk = {n: i for i, n in enumerate(dat["norm_key"])}
ph["ux"] = ph["norm"].str.casefold().map(nk)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype="auto", device_map="cuda")
model.eval()


def centroid(g):
    v = (X[g["ux"].to_numpy()] * g["weight"].to_numpy()[:, None]).sum(0)
    return v / np.linalg.norm(v)


def top_phrases(g, n):
    return g.groupby("phrase")["weight"].sum().nlargest(n)


LOG = TAXONOMY_FOLDER / "status" / "codebook_draft.jsonl"
rows = []
for kind, K, col, st in (("topic", K_TOPIC, "topic_c", stab_t), ("theme", K_THEME, "theme_c", stab_h)):
    groups = {c: ph[ph[col] == c] for c in range(1, K + 1)}
    groups = {c: g for c, g in groups.items() if len(g)}
    ids = list(groups)
    C = np.stack([centroid(g) for g in groups.values()])
    sims = C @ C.T
    np.fill_diagonal(sims, -1)
    for i, c in enumerate(ids):
        nb_ids = [ids[j] for j in np.argsort(sims[i])[::-1][:2]]
        neighbors = [list(top_phrases(groups[nn], 8).index) for nn in nb_ids]
        torch.manual_seed(int.from_bytes(hashlib.sha256(f"{kind}:{c}".encode()).digest()[:4], "big"))
        tp = top_phrases(groups[c], 30)
        members = "\n".join(f"[{j}] {p} — {int(w)}" for j, (p, w) in enumerate(tp.items(), 1))
        user = CB_USER.format(members=members, n1="; ".join(neighbors[0][:8]), n2="; ".join(neighbors[1][:8]))
        log = CallLog(LOG, f"{kind}:{c}", MODEL, prompt_version=CODEBOOK_PROMPT_VERSION)
        t0 = time.time()
        out, hit_cap = generate(model, tokenizer, CB_SYSTEM, user, max_new_tokens=MAX_NEW_TOKENS)
        log.write("codebook", 0, 0, 1, user, out, tokenizer, time.time() - t0)
        entry, errors = parse_entry(out, {p.casefold() for p in tp.index})
        if hit_cap:
            errors.append("hit_token_cap")
        srow = st.loc[c] if c in st.index else None
        rows.append({
            "kind": kind, "cluster": int(c),
            "stability": float(srow["stability"]) if srow is not None else np.nan,
            "cohesion": float(srow["cohesion"]) if srow is not None else np.nan,
            "weight": float(groups[c]["weight"].sum()), "n_phrases": int(groups[c]["phrase"].nunique()),
            "top_members": json.dumps(list(tp.index[:10])), "entry": json.dumps(entry) if entry else None, "errors": ";".join(errors),
        })
    print(f"{kind}: {len(ids)} entries drafted")

draft_df = pd.DataFrame(rows)
draft_df.to_parquet(TAXONOMY_FOLDER / "codebook_draft.parquet", index=False)
ok = draft_df["entry"].notna()
print(f"clean {ok.sum()}/{len(draft_df)}; errors on {(draft_df['errors'] != '').sum()}")


topic: 150 entries drafted


theme: 15 entries drafted
clean 165/165; errors on 5


## Step 7. Write the draft as a document

The document lists every topic under its plurality theme with its label, definition, examples, diagnostics, and top members. It is the document the review of T2.1.4 was made on. A CSV copy holds the same fields.


In [6]:
draft_df = pd.read_parquet(TAXONOMY_FOLDER / "codebook_draft.parquet")
themes = draft_df[draft_df["kind"] == "theme"].set_index("cluster")
topics = draft_df[draft_df["kind"] == "topic"]
tt = {int(t): int(nest.loc[t, "theme_c"]) for t in nest.index}

lines = ["# Codebook draft: 150 topic clusters under 15 broad clusters, for review", "",
         "Stability and cohesion are diagnostics. The review in T2.1.4 decides KEEP, MERGE, or DROP for each entry.", ""]
for th in sorted(themes.index):
    e = json.loads(themes.loc[th, "entry"]) if themes.loc[th, "entry"] else None
    lines += [f"## Broad cluster {th}: {e['label'] if e else '(draft failed)'}", ""]
    for r in topics.itertuples():
        if tt.get(r.cluster) != th:
            continue
        t = json.loads(r.entry) if r.entry else None
        lines += [f"### Topic {r.cluster}: {t['label'] if t else '(failed: ' + r.errors + ')'}",
                  f"- stability {r.stability:.2f} | cohesion {r.cohesion:.2f} | weight {r.weight:.0f} | {r.n_phrases} phrases"]
        if t:
            lines += [f"- definition: {t['definition']}", "- examples: " + "; ".join(t["exemplars"])]
        lines += ["- members: " + "; ".join(json.loads(r.top_members)[:8]), ""]

TABLE_FOLDER.mkdir(parents=True, exist_ok=True)
(TABLE_FOLDER / "table_codebook_draft.md").write_text("\n".join(lines), encoding="utf-8")
flat = draft_df.assign(theme_cluster=lambda d: d["cluster"].map(tt).where(d["kind"] == "topic"))
flat = pd.concat([flat.drop(columns=["entry"]),
                  flat["entry"].map(lambda e: json.loads(e) if e else {}).apply(pd.Series)], axis=1)
flat.to_csv(TABLE_FOLDER / "table_codebook_draft.csv", index=False)
print(f"wrote table_codebook_draft.md ({len(lines):,} lines) and table_codebook_draft.csv ({len(flat)} rows)")


wrote table_codebook_draft.md (934 lines) and table_codebook_draft.csv (165 rows)


## Step 8. Check the draft


In [7]:
ok = draft_df["entry"].notna()
print(f"1) drafted : {ok.sum()}/{len(draft_df)} entries parsed")
print(f"2) lint    : {sum(bool(json.loads(r.entry).get('label_lint')) for r in draft_df[ok].itertuples())} labels carry a directional word, for review")
print(f"3) errors  : {draft_df.loc[draft_df['errors'] != '', 'errors'].str.split(';').explode().value_counts().to_dict()}")


1) drafted : 165/165 entries parsed
2) lint    : 7 labels carry a directional word, for review
3) errors  : {'exemplars_not_verbatim:5': 5}
